Step 1 — Imports

In [7]:
# Core
import pandas as pd
import numpy as np
import joblib

# Sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Optional (recommended)
from xgboost import XGBClassifier

Step 2 — Load Data

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
default_of_credit_card_clients = fetch_ucirepo(id=350) 
  
# data (as pandas dataframes) 
X = default_of_credit_card_clients.data.features 
y = default_of_credit_card_clients.data.targets 
  
# metadata 
print(default_of_credit_card_clients.metadata) 
  
# variable information 
print(default_of_credit_card_clients.variables) 


{'uci_id': 350, 'name': 'Default of Credit Card Clients', 'repository_url': 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients', 'data_url': 'https://archive.ics.uci.edu/static/public/350/data.csv', 'abstract': "This research aimed at the case of customers' default payments in Taiwan and compares the predictive accuracy of probability of default among six data mining methods.", 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 30000, 'num_features': 23, 'feature_types': ['Integer', 'Real'], 'demographics': ['Sex', 'Education Level', 'Marital Status', 'Age'], 'target_col': ['Y'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C55S3H', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 365, 'type': 'NATIVE', 'title': 'The comparisons of data mining techniques for the predictive accuracy of 

In [3]:
X

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23
0,20000,2,2,1,24,2,2,-1,-1,-2,...,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,1,39,0,0,0,0,0,...,208365,88004,31237,15980,8500,20000,5003,3047,5000,1000
29996,150000,1,3,2,43,-1,-1,-1,-1,0,...,3502,8979,5190,0,1837,3526,8998,129,0,0
29997,30000,1,2,2,37,4,3,2,-1,0,...,2758,20878,20582,19357,0,0,22000,4200,2000,3100
29998,80000,1,3,1,41,1,-1,0,0,0,...,76304,52774,11855,48944,85900,3409,1178,1926,52964,1804


In [4]:
y

,Y
0,1
1,1
2,0
3,0
4,0
...,...
29995,0
29996,0
29997,1
29998,1


Step 3 — Train/Test Split (HOLDOUT SET)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (24000, 26)
Test shape: (6000, 26)


Step 4 — Define Models + Hyperparameter Grids

In [22]:
models = {
    "LogisticRegression": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),
        {
            "model__C": [0.01, 0.1, 1, 10],
            "model__class_weight": [None, "balanced"]
        }
    ),
    
    "RandomForest": (
        Pipeline([
            ("model", RandomForestClassifier(random_state=42))
        ]),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 5]
        }
    ),
    
    "XGBoost": (
        Pipeline([
            ("model", XGBClassifier(
                eval_metric="logloss",
                use_label_encoder=False,
                random_state=42
            ))
        ]),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 5],
            "model__learning_rate": [0.01, 0.1]
        }
    )
}

Step 5 — Hyperparameter Tuning (Validation via CV)

In [23]:
best_models = {}
results = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, (pipeline, param_grid) in models.items():
    
    print(f"\n🔍 Tuning {name}...")
    
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="roc_auc",   # Important for credit default
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_models[name] = grid_search.best_estimator_
    
    results.append({
        "Model": name,
        "Best CV Score (ROC-AUC)": grid_search.best_score_,
        "Best Params": grid_search.best_params_
    })

results_df = pd.DataFrame(results).sort_values(
    by="Best CV Score (ROC-AUC)",
    ascending=False
)

results_df


🔍 Tuning LogisticRegression...
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept



🔍 Tuning RandomForest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarni


🔍 Tuning XGBoost...
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:59:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:59:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:59:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nipunborji/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:59:58] WARNING: /Users/runner/work/xgboost/xgbo

,Model,Best CV Score (ROC-AUC),Best Params
2,XGBoost,0.782676,"{'model__learning_rate': 0.1, 'model__max_dept..."
1,RandomForest,0.782179,"{'model__max_depth': 10, 'model__min_samples_s..."
0,LogisticRegression,0.726432,"{'model__C': 0.01, 'model__class_weight': 'bal..."


Step 6 — Evaluate Best Model on TEST SET

In [24]:
# Select best model based on CV score
best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

print(f"\n🏆 Best Model: {best_model_name}")

# Predictions
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# Metrics
print("\nTest Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


🏆 Best Model: XGBoost

Test Performance:
Accuracy: 0.8186666666666667
Precision: 0.6685472496473907
Recall: 0.3571966842501884
F1 Score: 0.4656188605108055
ROC-AUC: 0.779837789310911

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4673
           1       0.67      0.36      0.47      1327

    accuracy                           0.82      6000
   macro avg       0.75      0.65      0.68      6000
weighted avg       0.80      0.82      0.80      6000



Step 7 — Save Best Model

In [25]:
joblib.dump(best_model, "best_credit_default_model.pkl")
print("✅ Model saved successfully.")

✅ Model saved successfully.


Step 8 — Load Model for Inference

In [26]:
loaded_model = joblib.load("best_credit_default_model.pkl")

Step 9 — Inference on New Data

In [27]:
def predict_default(input_dataframe):
    """
    input_dataframe must have same columns as X
    """
    prediction = loaded_model.predict(input_dataframe)
    probability = loaded_model.predict_proba(input_dataframe)[:, 1]
    
    return {
        "Prediction (1=Default)": int(prediction[0]),
        "Default Probability": float(probability[0])
    }

In [28]:
sample = X_test.iloc[[0]]
predict_default(sample)

{'Prediction (1=Default)': 0, 'Default Probability': 0.136433407664299}

With Feature Engineering

In [20]:
X["TOTAL_BILL"] = X[["X12","X13","X14","X15","X16","X17"]].sum(axis=1)
X["TOTAL_PAYMENT"] = X[["X18","X19","X20","X21","X22","X23"]].sum(axis=1)
X["PAYMENT_RATIO"] = X["TOTAL_PAYMENT"] / (X["TOTAL_BILL"] + 1)

In [19]:
X

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X17,X18,X19,X20,X21,X22,X23,TOTAL_BILL,TOTAL_PAYMENT,PAYMENT_RATIO
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,689,0,0,0,0,7704,689,0.089422
1,120000,2,2,2,26,-1,2,0,0,0,...,3261,0,1000,1000,1000,0,2000,17077,5000,0.292774
2,90000,2,2,2,34,0,0,0,0,0,...,15549,1518,1500,1000,1000,1000,5000,101653,11018,0.108387
3,50000,2,2,1,37,0,0,0,0,0,...,29547,2000,2019,1200,1100,1069,1000,231334,8388,0.036259
4,50000,1,2,1,57,-1,0,-1,0,0,...,19131,2000,36681,10000,9000,689,679,109339,59049,0.540049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,1,39,0,0,0,0,0,...,15980,8500,20000,5003,3047,5000,1000,725349,42550,0.058661
29996,150000,1,3,2,43,-1,-1,-1,-1,0,...,0,1837,3526,8998,129,0,0,21182,14490,0.684039
29997,30000,1,2,2,37,4,3,2,-1,0,...,19357,0,0,22000,4200,2000,3100,70496,31300,0.443991
29998,80000,1,3,1,41,1,-1,0,0,0,...,48944,85900,3409,1178,1926,52964,1804,266611,147181,0.552042
